# KOTFlash × TinyStories: A/B training benchmark (4096 tokens, 100 steps)

This notebook runs the same small GPT-style language model twice on the same pre-tokenized TinyStories batches: expanded-KV PyTorch SDPA vs. KOTFlash Triton. Both start from identical weights; backend order is shuffled per paired step.

**Runtime:** select a Tesla T4 GPU in Colab. The notebook refuses to label another GPU as a T4 result. It uses Python 3.12.16, PyTorch 2.6.0+cu124 and Triton 3.2.x.

The dataset is `skeskinen/TinyStories-hf`, which has individual story examples and a separate validation split. Data loading/tokenization is excluded from GPU timing. The model is a 2-layer, 512-hidden GPT-style proxy, not the full KOT-V2-2B. It measures real next-token LM training, not only attention.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, **kwargs)

run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
UV = shutil.which("uv")
if UV is None:
    candidate = Path(sys.prefix) / "bin" / "uv"
    if candidate.is_file():
        UV = str(candidate)
if UV is None:
    raise RuntimeError("uv was installed but its executable was not found on PATH")

run([UV, "python", "install", "3.12.16"])
ENV_DIR = Path("/content/kotflash-env")
run([UV, "venv", "--seed", "--python", "3.12.16", ENV_DIR])
PY = str(ENV_DIR / "bin" / "python")
run([PY, "-m", "pip", "--version"])
print("Isolated interpreter:", PY)


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO = Path("/content/KOTFlash")
BRANCH = "main"
if REPO.exists() and not (REPO / ".git").exists():
    shutil.rmtree(REPO)
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", "https://github.com/AitiKot/KOTFlash.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

run([PY, "-m", "pip", "install", "--upgrade", "pip"])
run([PY, "-m", "pip", "install", "--index-url", "https://download.pytorch.org/whl/cu124", "torch==2.6.0"])
run([PY, "-m", "pip", "install", "numpy", "pytest", "build", "ninja", "datasets>=3,<5", "transformers>=4.45,<5"])
run([PY, "-m", "pip", "install", "-e", ".[t4,test]"], cwd=REPO)
print("Repository:", REPO)


In [ ]:
# Verify the GPU and exact software stack before loading data.
check = r'''import torch, triton, kotflash
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA unavailable")
print("Capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)
print("KOTFlash:", kotflash.__file__)
assert torch.cuda.is_available(), "Enable a GPU runtime in Colab"
assert torch.cuda.get_device_capability(0) == (7, 5), "This benchmark requires SM75 / Tesla T4"
assert "T4" in torch.cuda.get_device_name(0), "Refusing to label another GPU as T4"
assert torch.__version__.split("+")[0] == "2.6.0" and torch.version.cuda == "12.4"
assert triton.__version__.split(".")[:2] == ["3", "2"]
'''
subprocess.run([PY, "-c", check], check=True)

# Run the static/unit suite before spending time on the benchmark.
env = os.environ.copy()
env["PATH"] = str(Path(PY).parent) + os.pathsep + env.get("PATH", "")
subprocess.run([PY, "-m", "pytest", "-q"], cwd=REPO, env=env, check=True)


In [ ]:
# 5 warmup + 100 measured steps per backend at context length 4096.
OUT = REPO / "artifacts" / "tinystories-4096-100step.json"
subprocess.run([
    PY, str(REPO / "benchmarks" / "benchmark_tinystories_training.py"),
    "--device", "cuda", "--seq", "4096", "--batch", "1",
    "--layers", "2", "--q-heads", "8", "--kv-heads", "2",
    "--head-dim", "64", "--steps", "100", "--warmup", "5",
    "--output", str(OUT),
], cwd=REPO, env=env, check=True)
print("Saved result:", OUT)


In [ ]:
# Package the result and exact benchmark source for download to your PC.
import zipfile
from datetime import datetime
from google.colab import files

archive = Path("/content") / f"KOTFlash_TinyStories_4096_100step_{datetime.now().strftime('%Y%m%d-%H%M%S')}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    z.write(OUT, arcname="tinystories-4096-100step.json")
    z.write(REPO / "benchmarks" / "benchmark_tinystories_training.py", arcname="benchmark_tinystories_training.py")
print("Archive:", archive, "size MB:", round(archive.stat().st_size / 1024 / 1024, 2))
files.download(str(archive))


## How to read the result

- `speedup_triton_vs_expanded_sdpa > 1.0`: KOTFlash used less total GPU time for the 100 measured optimizer steps.
- Compare each backend's `sum_gpu_step_ms`, `median_step_ms` and `training_tokens_per_second`.
- Check `initial_eval_loss_parity_passed` and `finite_after_training`; invalid numerical results must not be used as speed claims.
- Per-backend times are CUDA-event measurements around LM forward, loss, backward and AdamW. Dataset preparation and kernel compilation warmup are excluded.

The ZIP contains the JSON report and benchmark source. Keep it together with the recorded GPU and software versions. This small proxy is not a measurement of KOT-V2-2B.
